# Druggability Axis Analysis

### Goal
Identify existing therapeutic compounds that target the broadly and strictly conserved pan-cancer metastatic metabolic genes.

### Purpose
To cross-reference the highly conserved metastatic metabolic signature against DGIdb to find druggable vulnerabilities. This helps prioritize targets for potential drug repurposing.

### Interpretation
- **Druggable Targets:** Genes with known drug interactions.
- **Clinical Relevance:** Overlap with FDA-approved drugs indicates immediate translational potential for treating metastasis.

### Inputs / Parameters
*Explicitly documented for traceability and reproducibility.*

**Configuration Files:**
- `pan_cancer_config.py`
- `pan_cancer_config.py`


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

%load_ext autoreload
%autoreload 2

from IPython.display import display, HTML
from dynamic_genes import get_dynamic_genes
TARGET_GENES = get_dynamic_genes('.')
from pan_cancer_config import ANALYSIS_SUFFIX
OUTPUT_DIR = '../output/druggability'
OUTPUT_BASENAME = f'druggability_axis{ANALYSIS_SUFFIX}'
from query_dbs import compile_drug_databases
from query_depmap import analyze_depmap_synergy
from query_advanced_analysis import query_tractability, query_string_ppi

In [2]:
# ==========================================
# ⚙️ USER PARAMETERS (Export Options)
# ==========================================
# Full Notebook HTML Report Export Toggle:
# - True  -> Automatically exports the entire notebook (Markdown, Code, Tables, Figures) to styled HTML
# - False -> Disables automatic HTML export
SAVE_AS_HTML = True  # DEFAULT: False. Change to True to export the entire notebook!

print(f"Analyzing Axis: {TARGET_GENES}")
print(f"Output Base: {OUTPUT_BASENAME}")

Analyzing Axis: ['PTGDR', 'SLC5A5', 'CCR9', 'CD1D', 'CNR2', 'CD48', 'GCNT4', 'AGER', 'CYP2E1', 'KLRG1', 'KCTD8', 'CYP19A1']
Output Base: druggability_axis_6MetCan_500k


## 2. Cross-reference Genes against Drug Databases

**Purpose**: Identifies if the genes in the hypothesized axis already have FDA-approved drugs or compounds in clinical trials.  
**Interpretation**: If a gene is druggable, we can rapidly test repurposing existing drugs. If multiple genes in the axis are druggable, it opens the door to combination therapy.

In [3]:
df_drugs = compile_drug_databases(TARGET_GENES)

if not df_drugs.empty:
    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_BASENAME}_drug_targets.csv")
    df_drugs.to_csv(csv_path, index=False)
    print(f"Saved drug targets to {csv_path}")
    display(df_drugs.head(15))
else:
    print("No drug interactions found.")

[DGIdb] Querying interactions for 12 genes...
[DGIdb] Found 140 interactions.
[OpenTargets] Resolving gene symbols to Ensembl IDs for 12 genes...
[OpenTargets] Resolved IDs: {'PTGDR': 'ENSG00000168229', 'SLC5A5': 'ENSG00000105641', 'CCR9': 'ENSG00000173585', 'CD1D': 'ENSG00000158473', 'CNR2': 'ENSG00000188822', 'CD48': 'ENSG00000117091', 'GCNT4': 'ENSG00000176928', 'AGER': 'ENSG00000204305', 'CYP2E1': 'ENSG00000130649', 'KLRG1': 'ENSG00000139187', 'KCTD8': 'ENSG00000183783', 'CYP19A1': 'ENSG00000137869'}
[OpenTargets] Fetching known drugs for PTGDR (ENSG00000168229)...
[OpenTargets] Fetching known drugs for SLC5A5 (ENSG00000105641)...
[OpenTargets] Fetching known drugs for CCR9 (ENSG00000173585)...
[OpenTargets] Fetching known drugs for CD1D (ENSG00000158473)...
[OpenTargets] Fetching known drugs for CNR2 (ENSG00000188822)...
[OpenTargets] Fetching known drugs for CD48 (ENSG00000117091)...
[OpenTargets] Fetching known drugs for GCNT4 (ENSG00000176928)...
[OpenTargets] Fetching known dr

,Database,Target_Gene,Drug_Name,Interaction_Type,Sources,Approval_Status,Indication_Disease
0,DGIdb,CYP2E1,ISOFLAVONE,Targeted,DGIdb GraphQL,N/A,NaN
1,DGIdb,CYP2E1,BENZODIAZEPINE,Targeted,DGIdb GraphQL,N/A,NaN
2,DGIdb,CYP2E1,BRADANICLINE,Targeted,DGIdb GraphQL,N/A,NaN
3,DGIdb,CYP2E1,SEVOFLURANE,Targeted,DGIdb GraphQL,N/A,NaN
4,DGIdb,CYP2E1,CYTARABINE,Targeted,DGIdb GraphQL,N/A,NaN
5,DGIdb,CYP2E1,PHENETHYL ISOCYANATE,Targeted,DGIdb GraphQL,N/A,NaN
6,DGIdb,CYP2E1,POLYETHYLENE GLYCOL 4000,Targeted,DGIdb GraphQL,N/A,NaN
7,DGIdb,CYP2E1,CHLORPROMAZINE,Targeted,DGIdb GraphQL,N/A,NaN
8,DGIdb,CYP2E1,VALERIAN ROOT EXTRACT,Targeted,DGIdb GraphQL,N/A,NaN
9,DGIdb,CYP2E1,ADJUVANT,Targeted,DGIdb GraphQL,N/A,NaN


## 3. Target Tractability Prediction (Open Targets)

**Purpose**: For targets lacking known drugs, we query computational models to predict if they are structurally "druggable" by small molecules or antibodies.  
**Interpretation**: High tractability scores indicate that a gene is a biologically viable target for *de novo* drug discovery, even if no drugs currently exist.

In [4]:
df_trac = query_tractability(TARGET_GENES)
if not df_trac.empty:
    display(df_trac)
else:
    print("No tractability data found.")

[Tractability] Querying Open Targets for tractability of ['PTGDR', 'SLC5A5', 'CCR9', 'CD1D', 'CNR2', 'CD48', 'GCNT4', 'AGER', 'CYP2E1', 'KLRG1', 'KCTD8', 'CYP19A1']...
[Tractability] Found tractability data for 12 genes.
[Tractability] Data Sources -> {'Cache': 12}


,Target_Gene,Small_Molecule_Tractable,Antibody_Tractable,Other_Modalities_Tractable,Source
0,PTGDR,False,False,False,Cache
1,SLC5A5,False,False,False,Cache
2,CCR9,False,False,False,Cache
3,CD1D,False,False,False,Cache
4,CNR2,False,False,False,Cache
5,CD48,False,False,False,Cache
6,GCNT4,False,False,False,Cache
7,AGER,False,False,False,Cache
8,CYP2E1,False,False,False,Cache
9,KLRG1,False,False,False,Cache


## 4. Protein-Protein Interaction (STRING Database)

**Purpose**: Queries the STRING network to check if there is physical or functional evidence that these targets interact with each other.  
**Interpretation**: High combined scores (>0.4) strongly support the "axis" hypothesis, indicating these genes operate in the same complex or pathway rather than in isolation.

In [5]:
df_ppi = query_string_ppi(TARGET_GENES)
if not df_ppi.empty:
    display(df_ppi)
else:
    print("No PPI network connections found between these genes.")

[STRING PPI] Querying network for: ['PTGDR', 'SLC5A5', 'CCR9', 'CD1D', 'CNR2', 'CD48', 'GCNT4', 'AGER', 'CYP2E1', 'KLRG1', 'KCTD8', 'CYP19A1']
[STRING PPI] Loading cached network from version-controlled cache: /Users/sakuramaezono/Library/CloudStorage/OneDrive-YokohamaCityUniversity/Personal/05_Python_repositories/metabConnectomeDB/input/api_cache/string_ppi_06bad5b61dffa411920e4688b9cae05c.csv


,Gene_A,Gene_B,Combined_Score,Experimental_Score,Database_Score,Source
0,KLRG1,CCR9,0.404,0.000,0.000,Cache
1,KLRG1,CD48,0.426,0.047,0.000,Cache
2,CYP19A1,CYP2E1,0.590,0.052,0.172,Cache


## 5. Synergistic Cell Death (DepMap Co-dependency)

**Purpose**: Uses Cancer Dependency Map (DepMap) CRISPR knockout data to calculate the correlation of gene effects across hundreds of cancer cell lines.  
**Interpretation**: 
- **Positive correlation (> 0.3)**: Suggests genes are in the same functional pathway; knocking out either has a similar effect.
- **Negative correlation**: Suggests potential synthetic lethality or compensatory mechanisms.

In [6]:
corr_matrix = analyze_depmap_synergy(TARGET_GENES)

if corr_matrix is not None:
    display(corr_matrix)
else:
    print("Skipping correlation visualization.")

[DepMap] Loading DepMap CRISPR data from /Users/sakuramaezono/Library/CloudStorage/OneDrive-YokohamaCityUniversity/Personal/05_Python_repositories/metabConnectomeDB/input/CRISPRGeneEffect.csv...
[DepMap] Found columns for: ['AGER', 'CCR9', 'CD1D', 'CD48', 'CNR2', 'CYP19A1', 'CYP2E1', 'GCNT4', 'KCTD8', 'KLRG1', 'PTGDR', 'SLC5A5']
[DepMap] Saved correlation plot to /Users/sakuramaezono/Library/CloudStorage/OneDrive-YokohamaCityUniversity/Personal/05_Python_repositories/metabConnectomeDB/output/druggability/druggability_axis_6MetCan_500k_depmap_correlation.png


,AGER,CCR9,CD1D,CD48,CNR2,CYP19A1,CYP2E1,GCNT4,KCTD8,KLRG1,PTGDR,SLC5A5
AGER,1.000000,0.006432,0.013300,0.035053,-0.001573,0.080385,-0.022631,-0.134349,0.069497,0.065352,-0.003100,-0.055001
CCR9,0.006432,1.000000,0.039304,0.021144,0.089528,0.011556,-0.058473,0.018182,-0.035475,-0.051169,0.045890,0.015215
CD1D,0.013300,0.039304,1.000000,0.110445,-0.036286,0.078828,0.012433,0.014477,0.055800,-0.012275,0.047522,0.054149
CD48,0.035053,0.021144,0.110445,1.000000,0.032843,0.019188,-0.042404,0.065694,0.024274,0.039782,0.019557,0.048150
CNR2,-0.001573,0.089528,-0.036286,0.032843,1.000000,0.000087,-0.003021,0.032163,-0.031001,0.002164,0.048472,-0.075753
CYP19A1,0.080385,0.011556,0.078828,0.019188,0.000087,1.000000,0.057386,-0.111444,0.112562,0.057027,0.041948,-0.008285
CYP2E1,-0.022631,-0.058473,0.012433,-0.042404,-0.003021,0.057386,1.000000,-0.043126,-0.015315,0.012896,-0.009508,-0.068639
GCNT4,-0.134349,0.018182,0.014477,0.065694,0.032163,-0.111444,-0.043126,1.000000,-0.143675,0.141364,0.064053,0.104968
KCTD8,0.069497,-0.035475,0.055800,0.024274,-0.031001,0.112562,-0.015315,-0.143675,1.000000,0.088623,0.036066,0.028769
KLRG1,0.065352,-0.051169,-0.012275,0.039782,0.002164,0.057027,0.012896,0.141364,0.088623,1.000000,0.055311,-0.012203


---

## 6. Export Full Notebook Report to HTML

Compiling this entire interactive notebook into a single publication-ready and highly interactive HTML report.

---
## Druggability of Pan-Cancer Conserved Genes
In addition to the specific GLS axis, we also query the DGIdb database for the strictly conserved **strictly conserved pan-cancer genes** (upregulated in all evaluated cancers) and the broadly conserved **broadly conserved genes** (upregulated in $\ge$ N-1 cancers).


In [7]:
import sys
if '..' not in sys.path: sys.path.append('..')
from pan_cancer_config import ANALYSIS_SUFFIX

df_23 = pd.read_csv(os.path.join(OUTPUT_DIR, f'druggable_targets_strictly_conserved{ANALYSIS_SUFFIX}.csv'))
print(f"Total drug interactions for strictly conserved genes: {len(df_23)}")
display(df_23.head(10))

Total drug interactions for strictly conserved genes: 2


,Gene,Drug,Database
0,CCR9,VERCIRNON,DGIdb
1,CCR9,MLN3126,DGIdb


In [8]:
import sys
if '..' not in sys.path: sys.path.append('..')
from pan_cancer_config import ANALYSIS_SUFFIX

df_181 = pd.read_csv(os.path.join(OUTPUT_DIR, f'druggable_targets_broadly_conserved{ANALYSIS_SUFFIX}.csv'))
print(f"Total drug interactions for broadly conserved genes: {len(df_181)}")
display(df_181.head(10))

Total drug interactions for broadly conserved genes: 137


,Gene,Drug,Database
0,CYP2E1,ISOFLAVONE,DGIdb
1,CYP2E1,BENZODIAZEPINE,DGIdb
2,CYP2E1,BRADANICLINE,DGIdb
3,CYP2E1,SEVOFLURANE,DGIdb
4,CYP2E1,CYTARABINE,DGIdb
5,CYP2E1,PHENETHYL ISOCYANATE,DGIdb
6,CYP2E1,POLYETHYLENE GLYCOL 4000,DGIdb
7,CYP2E1,CHLORPROMAZINE,DGIdb
8,CYP2E1,VALERIAN ROOT EXTRACT,DGIdb
9,CYP2E1,ADJUVANT,DGIdb
